### ── Reproduce the original typology (Methodology §3.8) ──

Step 1: CFI Churn median split

Step 2: Net Cascade quartiles among high-churn MSOAs

In [1]:
from scipy import stats

In [ ]:
for yr in ['11', '21']:
    churn_col = f'CFI_Churn_{yr}'
    nc_col = f'Net_Cascade_{yr}'
 
    churn_med = df[churn_col].median()
    high_churn = df[df[churn_col] >= churn_med]
    nc_q25 = high_churn[nc_col].quantile(0.25)
    nc_q75 = high_churn[nc_col].quantile(0.75)
 
    def classify(row, _churn_col=churn_col, _nc_col=nc_col,
                 _churn_med=churn_med, _nc_q25=nc_q25, _nc_q75=nc_q75):
        if row[_churn_col] < _churn_med:
            return 'Stable'
        nc = row[_nc_col]
        if nc >= _nc_q75:
            return 'Ascent-Dominated'
        elif nc <= _nc_q25:
            return 'Displacement-Dominated'
        else:
            return 'Cascading Circulation'
 
    df[f'OrigTypology_{yr}'] = df.apply(classify, axis=1)
 
    print(f'=== 20{yr} Thresholds ===')
    print(f'  CFI Churn median: {churn_med:.0f}')
    print(f'  Net Cascade Q25 (high-churn): {nc_q25:.0f}')
    print(f'  Net Cascade Q75 (high-churn): {nc_q75:.0f}')
    print(f'  Distribution:')
    print(f'  {df[f"OrigTypology_{yr}"].value_counts().to_string()}')
    print()

In [ ]:
# ── Validation: Original typology vs IMD_Pctile_Change ──
print('=' * 65)
print('VALIDATION: Original Typology vs IMD Percentile Change')
print('=' * 65)
 
order = ['Ascent-Dominated', 'Cascading Circulation',
         'Displacement-Dominated', 'Stable']
 
for yr in ['11', '21']:
    typ_col = f'OrigTypology_{yr}'
    print(f'\n--- 20{yr} Typology ---')
 
    # Group means
    print(f'{"Type":<28s} {"n":>5s} {"Mean IMD Pctile Chg":>20s} {"Median":>10s}')
    print('-' * 65)
    groups = []
    for t in order:
        subset = df[df[typ_col] == t]['IMD_Pctile_Change']
        groups.append(subset.values)
        print(f'{t:<28s} {len(subset):>5d} {subset.mean():>+20.4f} '
              f'{subset.median():>+10.4f}')
 
    # Kruskal-Wallis
    kw_stat, kw_p = stats.kruskal(*groups)
    print(f'\n  Kruskal-Wallis H = {kw_stat:.2f}, p = {kw_p:.2e}')
 
    # One-way ANOVA
    f_stat, f_p = stats.f_oneway(*groups)
    print(f'  One-way ANOVA  F = {f_stat:.2f}, p = {f_p:.2e}')
 
    # Pairwise: Ascent vs Displacement
    asc = df[df[typ_col] == 'Ascent-Dominated']['IMD_Pctile_Change']
    disp = df[df[typ_col] == 'Displacement-Dominated']['IMD_Pctile_Change']
    mw_stat, mw_p = stats.mannwhitneyu(asc, disp, alternative='two-sided')
    print(f'  Mann-Whitney (Ascent vs Displacement): '
          f'U = {mw_stat:.0f}, p = {mw_p:.2e}')
    print(f'    Ascent mean: {asc.mean():+.4f}, '
          f'Displacement mean: {disp.mean():+.4f}')
 
    # Pairwise: Active (all 3) vs Stable
    active = df[df[typ_col] != 'Stable']['IMD_Pctile_Change']
    stable = df[df[typ_col] == 'Stable']['IMD_Pctile_Change']
    mw_stat2, mw_p2 = stats.mannwhitneyu(active, stable, alternative='two-sided')
    print(f'  Mann-Whitney (Active vs Stable): '
          f'U = {mw_stat2:.0f}, p = {mw_p2:.2e}')
    print(f'    Active mean: {active.mean():+.4f}, '
          f'Stable mean: {stable.mean():+.4f}')